In [ ]:
from typing import (
    TypedDict,
    Literal,
    Annotated,
)

import operator
import os
from dotenv import load_dotenv
from pydantic import BaseModel

from langchain_openai import ChatOpenAI
from langchain_core.messages import (
    SystemMessage,
    HumanMessage,
    AIMessage,
)
from langchain_core.tools import tool
# from langchain.agents import create_agent

from langgraph.graph import (
    StateGraph,
    START,
    END,
    MessagesState,
)
from langgraph.graph.message import add_messages
from langgraph.types import (
    Command,
    Send,
)
from langgraph.checkpoint.memory import InMemorySaver

In [ ]:
load_dotenv()

In [ ]:
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY not found - add it to your .env file.")
    
model = ChatOpenAI(
    model="nvidia/nemotron-3.5-lightning:free",
    temperature=0,
    base_url="https://openrouter.ai/api/v1",
    max_retries=3,   # free models hit 429 rate limits often
    timeout=120,
)

In [ ]:
# agent state

class AgentState(TypedDict):
    Task: str
    message: Annotated[list, add_messages]
    output: str
    route: str

In [ ]:
# creating Specialist agents for Frontend teams

# react agent
def react_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are a React specialist. Handle all React component and frontend tasks.")
    response = model.invoke(system, state["message"])
    return {"messages": [response], "output": response.content}

# ui agent
def ui_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are a UI specialist. Build clean, accessible user interfaces and handle all UI releated task.")
    response = model.invoke(system, state["message"])
    return {"message": response, "output": response.content}

# Accessibility Agent
def accessibility_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are an Accessibility specialist. Ensure all UI components meet WCAG 2.1 AA standards: semantic HTML, keyboard navigation, screen reader support, color contrast, and proper ARIA usage.")
    response = model.invoke(system, state["message"])
    return {"message": response, "output": response.content}

In [ ]:
# creating Specialist agents for backend teams

# api agent
def api_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are an API specialist. Design RESTful APIs, handle routing, validation, serialization, and ensure consistent error responses.")
    response = model.invoke(system, state["message"])
    return {"message":[response], "output": response.content}

# auth agent
def auth_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are an Authentication specialist. Implement secure auth: JWT/OAuth2, token refresh, password hashing, MFA, session management, and RBAC.")
    response = model.invoke(system, state["message"])
    return {"message":[response], "output": response.content}

# microsoft agent
def microsoft_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are a Microsoft ecosystem specialist. Handle Azure services, Microsoft Graph, Entra ID, Power Platform, and Office 365 integrations.")
    response = model.invoke(system, state["message"])
    return {"messages": [response], "output": response.content}